In [ ]:
import numpy as np
import pandas as pd
import statistics
import matplotlib.pyplot as plt
from scipy.stats import linregress
from tqdm import tqdm

In [ ]:
genes = {}

genes_df = pd.read_csv('/mnt/project/DNA repair genes/repair_genes_map.txt', sep = ' ', header = None)

for _, row in genes_df.iterrows():
    if row[3] == 'X':
        continue
    genes[row[0]] = True

In [ ]:
difficult_regions = {}

with open('/mnt/project/DNM/giab_difficult_merged_oct27.bed') as f:
    
    next(f)
    
    for row in f:
        
        row = row.strip()
        row = row.split('\t')
        
        chrom = int(row[0])
        start = int(row[1])
        end = int(row[2])
        
        if chrom not in difficult_regions:
            difficult_regions[chrom] = []
        difficult_regions[chrom].append((start, end))

In [ ]:
in_difficult = {}

for _, row in tqdm(genes_df.iterrows()):
        
    if row[3] == 'X':
        continue

    gene = row[0]
    gene_chrom = int(row[3])
    gene_start = int(row[4])
    gene_end = int(row[5])
    gene_length = gene_end - gene_start + 1

    overlap_bp = 0

    for start, end in difficult_regions[gene_chrom]:
        overlap_start = max(gene_start, start)
        overlap_end = min(gene_end, end)
        if overlap_start <= overlap_end:
            overlap_bp += overlap_end - overlap_start + 1

    in_difficult[gene] = overlap_bp / gene_length
        
print(sum(in_difficult.values())/len(in_difficult))

In [ ]:
in_difficult = dict(sorted(in_difficult.items(), key = lambda x: (-x[1])))
fracts = np.array(list(in_difficult.values()))

Q1 = np.percentile(fracts, 25)
Q3 = np.percentile(fracts, 75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

filtered_genes = {gene: f for gene, f in in_difficult.items() if f <= upper_bound}

print(upper_bound)

In [ ]:
plt.hist(fracts, bins = 20)
plt.axvline(upper_bound, color = 'red', linestyle = '--', linewidth = 2, 
            label = f'Q3 + 1.5 * IQR = {upper_bound:.4f}')
plt.xlabel('Fraction of gene in difficult regions')
plt.ylabel('Frequency')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
filtered_genes_df = genes_df[genes_df[0].isin(filtered_genes)].copy()
filtered_genes_df.reset_index(drop = True, inplace = True)
filtered_genes_df.to_csv('repair_genes_map_filtered.txt', sep = ' ', header = False, index = False)